In [1]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

# Load cleaned CREDIT CARD data
with open("../data/clean/cc_data.pkl", "rb") as f:
    X_train, X_test, y_train, y_test = pickle.load(f)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Fraud in train:", y_train.sum())
print("Fraud in test:", y_test.sum())

Train shape: (227845, 30)
Test shape: (56962, 30)
Fraud in train: 394
Fraud in test: 98


In [2]:
X_train_rnn = X_train.values.reshape((X_train.shape[0], 1, X_train.shape[1]))
X_test_rnn = X_test.values.reshape((X_test.shape[0], 1, X_test.shape[1]))

print("RNN Train shape:", X_train_rnn.shape)
print("RNN Test shape:", X_test_rnn.shape)

RNN Train shape: (227845, 1, 30)
RNN Test shape: (56962, 1, 30)


In [3]:
rnn_model = Sequential([
    LSTM(64, input_shape=(1, X_train.shape[1])),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

rnn_model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
)

rnn_model.summary()

c:\Users\guerd\anaconda3\envs\credit_project\lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 64)             │        24,320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 26,433 (103.25 KB)

 Trainable params: 26,433 (103.25 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
early_stop = EarlyStopping(monitor='val_loss', patience=2, restore_best_weights=True)

history = rnn_model.fit(
    X_train_rnn, y_train,
    epochs=5,
    batch_size=512,
    validation_split=0.2,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 9s 12ms/step - accuracy: 0.9949 - auc: 0.8612 - loss: 0.0928 - val_accuracy: 0.9994 - val_auc: 0.9379 - val_loss: 0.0039
Epoch 2/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 5s 14ms/step - accuracy: 0.9994 - auc: 0.9276 - loss: 0.0039 - val_accuracy: 0.9994 - val_auc: 0.9392 - val_loss: 0.0032
Epoch 3/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 5s 13ms/step - accuracy: 0.9994 - auc: 0.9392 - loss: 0.0032 - val_accuracy: 0.9995 - val_auc: 0.9392 - val_loss: 0.0030
Epoch 4/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9994 - auc: 0.9460 - loss: 0.0030 - val_accuracy: 0.9994 - val_auc: 0.9512 - val_loss: 0.0029
Epoch 5/5
357/357 ━━━━━━━━━━━━━━━━━━━━ 4s 10ms/step - accuracy: 0.9994 - auc: 0.9477 - loss: 0.0028 - val_accuracy: 0.9994 - val_auc: 0.9451 - val_loss: 0.0029


In [5]:
y_proba_rnn = rnn_model.predict(X_test_rnn).flatten()
y_pred_rnn = (y_proba_rnn > 0.5).astype(int)

print(classification_report(y_test, y_pred_rnn))
print("AUC-ROC:", roc_auc_score(y_test, y_proba_rnn))

1781/1781 ━━━━━━━━━━━━━━━━━━━━ 5s 3ms/step
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     56864
           1       0.80      0.83      0.81        98

    accuracy                           1.00     56962
   macro avg       0.90      0.91      0.91     56962
weighted avg       1.00      1.00      1.00     56962

AUC-ROC: 0.9868633036360295


In [6]:
import os
os.makedirs("../models", exist_ok=True)
rnn_model.save("../models/agent1_rnn.h5")
print("Saved!")

Saved!
